
## Folder 04 — File 06: Silver Shipments:


**Transformation rules:**

- Trim shipment, order, carrier, and tracking identifiers.
- Standardize carrier and shipment status values.
- Convert shipped_date and delivered_date into timestamps.
- Validate shipment status against allowed values.
- Check delivery-date consistency.
- Quarantine invalid records.
- Deduplicate using shipment_id.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_shipments"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_shipments"
)

ALLOWED_SHIPMENT_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze shipments

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze shipment records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize shipment columns

cleaned_df = (
    bronze_df

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id").cast("string"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )
    .withColumn(
        "carrier",
        F.upper(F.trim(F.col("carrier").cast("string")))
    )
    .withColumn(
        "shipment_status",
        F.upper(F.trim(F.col("shipment_status").cast("string")))
    )
    .withColumn(
        "tracking_number",
        F.trim(F.col("tracking_number").cast("string"))
    )
    .withColumn(
        "shipped_date",
        F.expr("try_cast(shipped_date AS TIMESTAMP)")
    )
    .withColumn(
        "delivered_date",
        F.expr("try_cast(delivered_date AS TIMESTAMP)")
    )
)

print("Shipment standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid shipment records

invalid_condition = (
    F.col("shipment_id").isNull()
    | (F.col("shipment_id") == "")

    | F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("carrier").isNull()
    | (F.col("carrier") == "")

    | F.col("shipment_status").isNull()
    | (~F.col("shipment_status").isin(
        ALLOWED_SHIPMENT_STATUSES
    ))

    | F.col("shipped_date").isNull()

    # Delivered shipments must have a delivery date.

    | (
        (F.col("shipment_status") == "DELIVERED")
        & F.col("delivered_date").isNull()
    )

    # Non-delivered shipments must not have a delivery date.

    | (
        (F.col("shipment_status") != "DELIVERED")
        & F.col("delivered_date").isNotNull()
    )

    # Delivery cannot happen before shipment.

    | (
        F.col("delivered_date").isNotNull()
        & (F.col("delivered_date") < F.col("shipped_date"))
    )
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("shipment_id").isNull()
                | (F.col("shipment_id") == ""),
                F.lit("Missing shipment_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("carrier").isNull()
                | (F.col("carrier") == ""),
                F.lit("Missing carrier")
            ),

            F.when(
                F.col("shipment_status").isNull()
                | (~F.col("shipment_status").isin(
                    ALLOWED_SHIPMENT_STATUSES
                )),
                F.lit("Invalid shipment_status")
            ),

            F.when(
                F.col("shipped_date").isNull(),
                F.lit("Missing or invalid shipped_date")
            ),

            F.when(
                (F.col("shipment_status") == "DELIVERED")
                & F.col("delivered_date").isNull(),
                F.lit("Delivered shipment has no delivered_date")
            ),

            F.when(
                (F.col("shipment_status") != "DELIVERED")
                & F.col("delivered_date").isNotNull(),
                F.lit("Non-delivered shipment has delivered_date")
            ),

            F.when(
                F.col("delivered_date").isNotNull()
                & (F.col("delivered_date") < F.col("shipped_date")),
                F.lit("Delivery date is before shipment date")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid shipment records: {valid_df.count()}")
print(f"Invalid shipment records: {invalid_df.count()}")


## 7. Remove duplicate shipment records

deduplication_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver shipment records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_shipment_groups = (
    silver_df
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_df
    .filter(
        F.col("shipment_id").isNull()
        | F.col("order_id").isNull()
        | F.col("carrier").isNull()
        | F.col("shipment_status").isNull()
        | (~F.col("shipment_status").isin(
            ALLOWED_SHIPMENT_STATUSES
        ))
        | F.col("shipped_date").isNull()
        | (
            (F.col("shipment_status") == "DELIVERED")
            & F.col("delivered_date").isNull()
        )
        | (
            (F.col("shipment_status") != "DELIVERED")
            & F.col("delivered_date").isNotNull()
        )
        | (
            F.col("delivered_date").isNotNull()
            & (F.col("delivered_date") < F.col("shipped_date"))
        )
    )
    .count()
)

print("=" * 50)
print("SILVER SHIPMENT VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate shipment groups: {duplicate_shipment_groups}")
print(f"Invalid Silver records: {invalid_silver_records}")

if duplicate_shipment_groups > 0 or invalid_silver_records > 0:
    raise Exception("Silver shipment validation failed.")

print("Silver shipment validation passed.")


## 11. Shipment status analysis

display(
    silver_df
    .groupBy("shipment_status")
    .agg(
        F.count("*").alias("total_shipments")
    )
    .orderBy("shipment_status")
)


## 12. Carrier performance analysis

display(
    silver_df
    .groupBy("carrier")
    .agg(
        F.count("*").alias("total_shipments"),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias("delivered_shipments")
    )
    .withColumn(
        "delivery_rate_percentage",
        F.round(
            F.col("delivered_shipments")
            / F.col("total_shipments") * 100,
            2
        )
    )
    .orderBy(F.desc("delivery_rate_percentage"))
)


## 13. Calculate delivery duration

delivered_shipments_df = (
    silver_df
    .filter(
        (F.col("shipment_status") == "DELIVERED")
        & F.col("delivered_date").isNotNull()
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date("delivered_date"),
            F.to_date("shipped_date")
        )
    )
)

display(
    delivered_shipments_df
    .groupBy("carrier")
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.round(
            F.avg("delivery_days"),
            2
        ).alias("average_delivery_days")
    )
    .orderBy("average_delivery_days")
)


## 14. Display final Silver shipment data

display(
    silver_df
    .orderBy("shipped_date")
    .limit(20)
)


## 15. Final summary

print("=" * 50)
print("SHIPMENT SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)
